In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchaudio.transforms as T
from torchaudio import load
from torch.utils.data import DataLoader, Dataset, random_split
import kagglehub

In [2]:
path = kagglehub.dataset_download("andradaolteanu/gtzan-dataset-music-genre-classification")
root_path = os.path.join(path, 'Data/genres_original')
print("Path to dataset files:", path)

100%|██████████| 1.21G/1.21G [00:12<00:00, 107MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/andradaolteanu/gtzan-dataset-music-genre-classification/versions/1


In [3]:
genres = sorted(os.listdir(root_path))
label_to_index = {label: ind for ind, label in enumerate(genres)}

In [4]:
len(genres)

10

In [5]:
label_to_index = {label:ind for ind, label in enumerate(genres)}
label_to_index

{'blues': 0,
 'classical': 1,
 'country': 2,
 'disco': 3,
 'hiphop': 4,
 'jazz': 5,
 'metal': 6,
 'pop': 7,
 'reggae': 8,
 'rock': 9}

In [6]:
mel_transform = T.MelSpectrogram(
    sample_rate=22050,
    n_fft=2048,
    hop_length=512,
    n_mels=64
)
amp_to_db = T.AmplitudeToDB()

max_len = 1280

In [7]:
freq_mask = T.FrequencyMasking(freq_mask_param=15)
time_mask = T.TimeMasking(time_mask_param=35)

In [8]:
class GTZAN(Dataset):
    def __init__(self, root_path, transform, max_len, is_train=False):
        self.root_path = root_path
        self.transform = transform
        self.max_len = max_len
        self.is_train = is_train
        self.audios = []

        for genre in sorted(os.listdir(root_path)):
            genre_path = os.path.join(root_path, genre)
            if not os.path.isdir(genre_path):
                continue
            for audio in os.listdir(genre_path):
                if audio.endswith('.wav'):
                    audio_path = os.path.join(genre_path, audio)
                    # Пропуск битого файла
                    if 'jazz.00054.wav' in audio_path:
                        continue
                    self.audios.append((audio_path, genre))

    def __len__(self):
        return len(self.audios)

    def __getitem__(self, index):
        audio_path, genre = self.audios[index]
        waveform, sr = load(audio_path)

        # Перевод в моно
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        target_sr = self.transform.sample_rate
        if sr != target_sr:
            resample = T.Resample(orig_freq=sr, new_freq=target_sr)
            waveform = resample(waveform)

        # Мел-спектрограмма + перевод в децибелы
        spectrogram = self.transform(waveform).squeeze(0)
        spectrogram = amp_to_db(spectrogram)

        # Выравнивание по длине
        if spectrogram.shape[1] > self.max_len:
            spectrogram = spectrogram[:, :self.max_len]
        elif spectrogram.shape[1] < self.max_len:
            pad_len = self.max_len - spectrogram.shape[1]
            spectrogram = F.pad(spectrogram, (0, pad_len))

        # Аугментация данных (только во время обучения)
        if self.is_train:
            spectrogram = freq_mask(spectrogram)
            spectrogram = time_mask(spectrogram)
            # Случайный гауссов шум
            noise = torch.randn_like(spectrogram) * 0.005
            spectrogram = spectrogram + noise

        return spectrogram, label_to_index[genre]

In [9]:
train_dataset = GTZAN(root_path, mel_transform, max_len, is_train=True)
test_dataset = GTZAN(root_path, mel_transform, max_len, is_train=False)

In [10]:
train_size = int(len(train_dataset) * 0.8)
test_size = len(train_dataset) - train_size

In [11]:
train_data, _ = random_split(
    train_dataset, [train_size, test_size],
    generator=torch.Generator().manual_seed(42)
)
_, test_data = random_split(
    test_dataset, [train_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

In [12]:
train = DataLoader(train_data, batch_size=32, shuffle=True)
test = DataLoader(test_data, batch_size=32)

In [13]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [14]:
class CheckAudio(nn.Module):
    def __init__(self):
        super().__init__()
        self.first = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((8, 8))
        )
        self.second = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 8 * 8, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 10)
        )

    def forward(self, audio):
        if audio.dim() == 3:
            audio = audio.unsqueeze(1)
        audio = self.first(audio)
        audio = self.second(audio)
        return audio

In [15]:
model_CheckAudio = CheckAudio().to(device)

In [16]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_CheckAudio.parameters(), lr=0.001, weight_decay=1e-4)

In [17]:
for epoch in range(60):
    model_CheckAudio.train()
    total_loss = 0.0
    for x_batch, y_batch in train:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        y_pred = model_CheckAudio(x_batch)
        loss = loss_fn(y_pred, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train)
    print(f'Эпоха {epoch + 1}, Потери: {avg_loss:.4f}')

Эпоха 1, Потери: 2.1549
Эпоха 2, Потери: 1.8685
Эпоха 3, Потери: 1.7252
Эпоха 4, Потери: 1.5907
Эпоха 5, Потери: 1.5125
Эпоха 6, Потери: 1.4487
Эпоха 7, Потери: 1.3912
Эпоха 8, Потери: 1.3591
Эпоха 9, Потери: 1.2715
Эпоха 10, Потери: 1.2748
Эпоха 11, Потери: 1.2939
Эпоха 12, Потери: 1.1827
Эпоха 13, Потери: 1.1588
Эпоха 14, Потери: 1.1383
Эпоха 15, Потери: 1.1377
Эпоха 16, Потери: 1.1514
Эпоха 17, Потери: 1.1625
Эпоха 18, Потери: 1.0987
Эпоха 19, Потери: 1.0526
Эпоха 20, Потери: 1.0832
Эпоха 21, Потери: 1.0276
Эпоха 22, Потери: 0.9656
Эпоха 23, Потери: 0.9730
Эпоха 24, Потери: 0.9553
Эпоха 25, Потери: 0.9703
Эпоха 26, Потери: 0.9991
Эпоха 27, Потери: 1.0667
Эпоха 28, Потери: 1.0071
Эпоха 29, Потери: 0.9436
Эпоха 30, Потери: 0.9006
Эпоха 31, Потери: 0.9462
Эпоха 32, Потери: 0.9195
Эпоха 33, Потери: 0.8650
Эпоха 34, Потери: 0.9263
Эпоха 35, Потери: 0.8795
Эпоха 36, Потери: 0.8308
Эпоха 37, Потери: 0.8300
Эпоха 38, Потери: 0.8739
Эпоха 39, Потери: 0.8199
Эпоха 40, Потери: 0.8314
Эпоха 41,

In [18]:
model_CheckAudio.eval()
correct, total = 0, 0

with torch.no_grad():
    for x_batch, y_batch in test:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        y_pred = model_CheckAudio(x_batch)
        pred = torch.argmax(y_pred, dim=1)

        correct += (pred == y_batch).sum().item()
        total += y_batch.size(0)

accuracy = correct * 100 / total
print(f'Точность предположения модели: {accuracy:.2f}%')

Точность предположения модели: 78.00%


In [19]:
def get_accuracy(model_to_eval, data_loader):
    model_to_eval.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x_batch, y_batch in data_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            y_pred = model_to_eval(x_batch)
            prediction = torch.argmax(y_pred, dim=1)
            total += y_batch.size(0)
            correct += (prediction == y_batch).sum().item()
    return round(correct * 100 / total, 2)

print(f'train score: {get_accuracy(model_CheckAudio, train)}%')
print(f'test score: {get_accuracy(model_CheckAudio, test)}%')

train score: 85.61%
test score: 78.0%


In [23]:
torch.save(model_CheckAudio.state_dict(), 'model_CheckAudio_GTZAN_DatasetMusicGenreClassification.pth')
torch.save(genres, 'labels_GTZAN_DatasetMusicGenreClassification.pth')

from google.colab import files
files.download('model_CheckAudio_GTZAN_DatasetMusicGenreClassification.pth')
files.download('labels_GTZAN_DatasetMusicGenreClassification.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
print(genres)

['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']
